# Notebook V — Terrain Constraints
<hr>

## Overview
This notebook runs **Module V** — it applies terrain yield reduction factors (fc5) to the soil- and climate-adjusted yield maps from NB4. Terrain constraints account for slope steepness and rainfall erosivity (Fournier Index), which limit mechanised farming and increase soil erosion risk.

### Prerequisites
NB1 through NB4 must have been run first. This notebook reads:
- NB4 outputs: `data_output/NB4/soil_clim_adj_yield_maiz_rain.tif`, `_irr.tif`
- `data_input/climate/precipitation.npy` — monthly precipitation (mm/month)
- `data_input/PK_Slope.tif` — slope in % (derived from SRTM DEM)
- `maiz_terrain_constraints_rain/irr.xlsx` — terrain reduction factor lookup tables

### What this notebook produces (saved to `data_output/NB5/`)
| Output file | Description |
|---|---|
| `terr_soil_clim_adj_yield_maiz_rain.tif` | fc5-adjusted rainfed yield (kg/ha) |
| `terr_soil_clim_adj_yield_maiz_irr.tif` | fc5-adjusted irrigated yield (kg/ha) |
| `fc5_maiz_rain.tif` | Terrain reduction factor — rainfed (0–1) |
| `fc5_maiz_irr.tif` | Terrain reduction factor — irrigated (0–1) |
| `fournier_index.tif` | Fournier rainfall erosivity index |

Prepared by Geoinformatics Center, AIT — Pakistan adaptation by Saqib Ali
<hr>

### Step 1 — Connect to Google Drive (Colab only)

Mount Google Drive to access the repo at `/content/drive/MyDrive/PK-PyAEZ/`.

> **Skip** if running locally.

In [ ]:
# ── Google Colab setup ──────────────────────────────────────────────
# Run these steps ONLY when using Google Colab.

# Step 1: Mount Google Drive to access the repo
# from google.colab import drive
# drive.mount('/content/drive')

### Step 2 — Install dependencies (Colab only)

Installs GDAL, Numba, and all required packages, then installs `pyaez` in editable mode from the Drive copy.

> **Skip** if running locally with the conda environment already set up.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', '-q', 'gdal-bin', 'libgdal-dev', 'python3-gdal'],
                   capture_output=True)
    _ver = subprocess.check_output(['gdal-config', '--version']).decode().strip()
    subprocess.run(['pip', 'install', '-q',
                    f'GDAL=={_ver}', 'numba', 'openpyxl', 'rasterio', 'requests', 'scipy', 'pandas'],
                   capture_output=True)
    subprocess.run(['pip', 'install', '-q', '-e', '/content/drive/MyDrive/PK-PyAEZ'], capture_output=True)
    print(f"Colab setup complete — GDAL {_ver}")

### Step 3 — Import Python libraries

| Library | Purpose |
|---|---|
| `numpy` | Loads precipitation array; array operations |
| `matplotlib` | Plots Fournier Index and fc5 yield maps |
| `pandas` | Reads terrain constraint Excel lookup tables |
| `gdal` (osgeo) | Opens slope and yield GeoTIFFs; saves outputs |
| `os`, `sys` | File path and module path management |

`gdal.UseExceptions()` silences the GDAL 4.0 FutureWarning.

In [ ]:
'''import supporting libraries'''
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas
try:
    from osgeo import gdal
except:
    import gdal
gdal.UseExceptions()
import sys

### Step 4 — Set working directory

Sets CWD to the repo root so all `./data_input/` and `./data_output/` paths resolve correctly.

- **Colab**: `/content/drive/MyDrive/PK-PyAEZ`
- **Local**: `..` (one level up from `tutorials/`)

In [ ]:
'Set the working directory'
import sys, os

if 'google.colab' in sys.modules:
    work_dir = '/content/drive/MyDrive/PK-PyAEZ'
else:
    work_dir = '..'

os.chdir(work_dir)
sys.path.insert(0, os.getcwd())
os.getcwd()

### Step 5 — Create output folder

Creates `data_output/NB5/` if it does not already exist.

In [ ]:
import os
folder_path = './data_output/NB5/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print("Folder created successfully.")
else:
    print("Folder already exists.")

<hr>

## MODULE V — Terrain Constraints

### Step 6 — Initialise TerrainConstraints

In [ ]:
'''importing library'''

from pyaez import TerrainConstraints
terrain_constraints = TerrainConstraints.TerrainConstraints()

from pyaez import UtilitiesCalc
obj_utilities = UtilitiesCalc.UtilitiesCalc()

### Step 7 — Load input data

| Variable | File | Description |
|---|---|---|
| `precipitation` | `climate/precipitation.npy` | Monthly precipitation (mm/month), shape 12×87×108 |
| `slope_map` | `PK_Slope.tif` | Terrain slope in % — derived from SRTM 90m DEM |

The slope map determines the base fc5 value per pixel. Steep slopes reduce yield potential for mechanised cropping.

In [ ]:
'''reading climate and slope data'''
basepath = './data_input/PK_Admin.tif'
precipitation = np.load('./data_input/climate/precipitation.npy') # mm / month

slope_map = gdal.Open('./data_input/PK_Slope.tif').ReadAsArray() # Percentage Slope

### Step 8 — Load terrain reduction factors from Excel

Reads crop-specific terrain suitability ratings — one sheet for rainfed, one for irrigated.

| File | Water supply | Indexed by |
|---|---|---|
| `maiz_terrain_constraints_rain.xlsx` | Rainfed | Slope class × Fournier Index class |
| `maiz_terrain_constraints_irr.xlsx` | Irrigated | Slope class × Fournier Index class |

In [ ]:
terrain_constraints.importTerrainReductionSheet(irr_file_path='./data_input/maiz_terrain_constraints_irr.xlsx',
                                rain_file_path='./data_input/maiz_terrain_constraints_rain.xlsx')

### Step 9 — Load climate and terrain data into TerrainConstraints

Passes the monthly precipitation array and slope raster into the object. These are combined to compute the Fournier Index.

In [ ]:
'''passing climate and slope data'''

terrain_constraints.setClimateTerrainData(precipitation, slope_map)

### Step 10 — Calculate the Fournier Index (FI)

The **Fournier Index** measures rainfall erosivity — how much energy rainfall has to detach and transport soil.

$$FI = \sum_{m=1}^{12} \frac{p_m^2}{P}$$

where $p_m$ is monthly precipitation and $P$ is mean annual precipitation. Higher FI → more erosion risk → lower fc5 on steep slopes.

In [ ]:
'''calculation of Fournier index'''
terrain_constraints.calculateFI()

# extraction of Fournier index (FI) if required
fi = terrain_constraints.getFI()

#### Visualise Fournier Index

Higher values indicate greater rainfall erosivity. Expect high FI in the monsoon belt (KPK, AJK) and low FI in arid zones (Balochistan, Sindh).

In [ ]:
# Visualizing the outputs
plt.imshow(fi)
plt.colorbar()
plt.title('Fournier Index')

#### Save Fournier Index

Saved to `data_output/NB5/fournier_index.tif` for reference and reporting.

In [ ]:
# saving FI data
obj_utilities.saveRaster(basepath, r'./data_output/NB5/fournier_index.tif', fi)

### Step 11 — Load yield maps from NB4

Loads the soil- and climate-adjusted yield maps produced by NB4. fc5 will be applied to these.

In [ ]:
'''reading yield data'''
yield_map_rain = gdal.Open('./data_output/NB4/soil_clim_adj_yield_maiz_rain.tif').ReadAsArray()

### Step 12 — Apply terrain constraints (Rainfed)

`applyTerrainConstraints(yield_map, 'R')` looks up fc5 for each pixel by slope class × FI class, then scales the yield:

```
yield_adjusted = yield_input × fc5(slope_class, FI_class)
```

- **Output:** `yield_map_rain_m5` (kg/ha), `fc5_rain` (0–1 per pixel)
- **Saved to:** `data_output/NB5/terr_soil_clim_adj_yield_maiz_rain.tif`, `fc5_maiz_rain.tif`

In [ ]:
'''applying terrain constraints (Rainfed)'''

yield_map_rain_m5 = terrain_constraints.applyTerrainConstraints(yield_map_rain, 'R') # I: Irrigated, R: Rain-fed

## get classified output
# yield_map_rain_class_m5 = obj_utilities.classifyFinalYield(yield_map_rain_m5)

# getting the terrain reduction factor (fc5)
fc5_rain = terrain_constraints.getTerrainReductionFactor()

#### Visualise rainfed results

Three-panel comparison: original yield (NB4) / terrain-adjusted yield / fc5 factor map.

In [ ]:
'''visualize result'''

plt.figure(1, figsize=(22, 7))

plt.subplot(1, 3, 1)
plt.imshow(yield_map_rain, vmax=np.max([yield_map_rain_m5, yield_map_rain]))
plt.colorbar(shrink=0.8)
plt.title('Original Rainfed Yield (NB4)')

plt.subplot(1, 3, 2)
plt.imshow(yield_map_rain_m5, vmax=np.max([yield_map_rain_m5, yield_map_rain]))
plt.colorbar(shrink=0.8)
plt.title('Terrain-Adjusted Rainfed Yield')

plt.subplot(1, 3, 3)
plt.imshow(fc5_rain, vmax=1, vmin=0)
plt.colorbar(shrink=0.8)
plt.title('fc5 Terrain Reduction Factor (Rainfed)')

#### Save rainfed outputs

| File | Content |
|---|---|
| `terr_soil_clim_adj_yield_maiz_rain.tif` | Terrain + soil + climate adjusted rainfed yield (kg/ha) |
| `fc5_maiz_rain.tif` | Spatial fc5 map — rainfed |

These are the primary inputs to **NB6 (Economic Suitability)**.

In [ ]:
# saving rainfed outputs
obj_utilities.saveRaster(basepath, './data_output/NB5/terr_soil_clim_adj_yield_maiz_rain.tif', yield_map_rain_m5)
obj_utilities.saveRaster(basepath, './data_output/NB5/fc5_maiz_rain.tif', fc5_rain)

---
### Step 13 — Apply terrain constraints (Irrigated)

Same workflow as Step 12 using `'I'` for irrigated. Loads irrigated yield from NB4, applies fc5 from the irrigated Excel LUT.

- **Output:** `yield_map_irr_m5` (kg/ha), `fc5_irr` (0–1 per pixel)

In [ ]:
'''reading Irrigated yield data'''

yield_map_irr = gdal.Open('./data_output/NB4/soil_clim_adj_yield_maiz_irr.tif').ReadAsArray()

In [ ]:
'''applying terrain constraints (Irrigated)'''
yield_map_irr_m5 = terrain_constraints.applyTerrainConstraints(yield_map_irr, 'I') # I: Irrigated, R: Rain-fed

## get classified output
# yield_map_irr_class_m5 = obj_utilities.classifyFinalYield(yield_map_irr_m5)

# getting the terrain reduction factor (fc5)
fc5_irr = terrain_constraints.getTerrainReductionFactor()

#### Visualise irrigated results

Three-panel comparison for irrigated. fc5_irr may differ from fc5_rain in steep areas where irrigation infrastructure is penalised more heavily.

In [ ]:
'''visualize result'''

plt.figure(1, figsize=(22, 7))

plt.subplot(1, 3, 1)
plt.imshow(yield_map_irr, vmax=np.max([yield_map_irr_m5, yield_map_irr]))
plt.colorbar(shrink=0.8)
plt.title('Original Irrigated Yield (NB4)')

plt.subplot(1, 3, 2)
plt.imshow(yield_map_irr_m5, vmax=np.max([yield_map_irr_m5, yield_map_irr]))
plt.colorbar(shrink=0.8)
plt.title('Terrain-Adjusted Irrigated Yield')

plt.subplot(1, 3, 3)
plt.imshow(fc5_irr, vmax=1, vmin=0)
plt.colorbar(shrink=0.8)
plt.title('fc5 Terrain Reduction Factor (Irrigated)')

#### Save irrigated outputs

| File | Content |
|---|---|
| `terr_soil_clim_adj_yield_maiz_irr.tif` | Terrain + soil + climate adjusted irrigated yield (kg/ha) |
| `fc5_maiz_irr.tif` | Spatial fc5 map — irrigated |

In [ ]:
# saving irrigated outputs
obj_utilities.saveRaster(basepath, './data_output/NB5/terr_soil_clim_adj_yield_maiz_irr.tif', yield_map_irr_m5)
obj_utilities.saveRaster(basepath, './data_output/NB5/fc5_maiz_irr.tif', fc5_irr)

<hr>

### END OF MODULE V — Terrain Constraints

**Next step → NB6:** Economic suitability analysis using the final adjusted yields.

Inputs passed forward to NB6:
- `data_output/NB5/terr_soil_clim_adj_yield_maiz_rain.tif`
- `data_output/NB5/terr_soil_clim_adj_yield_maiz_irr.tif`

<hr>